# PySpark — Traitement distribué d'images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/04_traitement_images.ipynb)

Notebook conçu pour **Google Colab** — adapté aux débutants en data engineering.

**Ce que tu vas apprendre :**
- Charger des milliers d'images en parallèle avec `spark.read.format("binaryFile")`
- Extraire des métadonnées depuis les noms de fichiers (lazy vs action)
- Écrire une UDF Python (Pillow) exécutée sur les workers Spark
- Visualiser un échantillon d'images directement dans le notebook
- Redimensionner toutes les images de façon distribuée

**Dataset :** Oxford-IIIT Pet Dataset — ~7 300 photos de 37 races de chats et chiens.

In [ ]:
## Section 0 — Setup (Colab uniquement)
!pip install -q pyspark findspark pillow matplotlib
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools import (
    colab_setup,
    datasets,
    image_display,
    image_utils,
    spark_monitor,
    spark_utils,
)
from pyspark.sql import functions as F

spark, monitor, helper = setup_colab("PySpark - Traitement Images")

## 1. Télécharger le dataset

Le dataset **Oxford-IIIT Pet** contient ~7 300 photos de chats et chiens (37 races).  
`download_oxford_pets()` télécharge l'archive (~800 MB) et l'extrait dans `/content/oxford_pets/images/`.

> Le téléchargement peut prendre 2-3 minutes selon la connexion Colab.

In [ ]:
# Téléchargement et extraction du dataset
images_path = download_oxford_pets()
print(f"\nImages disponibles dans : {images_path}")

## 2. Chargement distribué — `binaryFile`

Spark peut lire n'importe quel fichier binaire (images, PDFs, audio…) via le format `binaryFile`.  
Il produit un DataFrame avec **4 colonnes automatiques** :

| Colonne | Type | Description |
|---|---|---|
| `path` | string | Chemin complet du fichier |
| `modificationTime` | timestamp | Date de dernière modification |
| `length` | long | Taille du fichier en octets |
| `content` | binary | Contenu brut du fichier (les bytes) |

> **Lazy evaluation :** `spark.read.format("binaryFile").load(...)` est **instantané** — Spark scanne juste le répertoire et construit un plan.  Aucune image n'est lue en mémoire avant une action (`count()`, `show()`…).  
> Vérifie dans SparkUI : aucun job ne doit apparaître après cette cellule.

In [ ]:
# Chargement lazy — aucune image lue avant une action
df_images = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.jpg") \
    .load(images_path)

df_images.printSchema()
print("\nPlan créé — aucun job visible dans SparkUI (lazy).")

## 3. Extraction de métadonnées depuis les noms de fichiers

Les noms de fichiers encodent des informations utiles :
- `Abyssinian_1.jpg` → Race: `Abyssinian`, ID: 1, Espèce: **Chat** (première lettre majuscule)
- `golden_retriever_12.jpg` → Race: `golden retriever`, ID: 12, Espèce: **Chien** (tout en minuscules)

On construit un **pipeline de transformations lazy** avec des expressions régulières pour extraire `race`, `species`, `image_id` et `size_mb` depuis la colonne `path`.

| Fonction Spark | Description | Exemple |
|---|---|---|
| `F.element_at(arr, -1)` | Dernier élément d'un tableau | `["a","b","c"]` → `"c"` |
| `F.regexp_replace(col, regex, val)` | Remplace par regex | `"_\d+\.jpg$"` → `""` |
| `F.regexp_extract(col, regex, groupe)` | Extrait un groupe regex | `"Abyssinian_1.jpg"` → `"1"` |
| `F.when(condition, val).otherwise(val)` | Équivalent `if/else` | majuscule → `"Cat"` |

> Toutes ces transformations sont **lazy** — Spark ne lit aucune image avant une action.

In [ ]:
from pyspark.sql import functions as F

# Extraire le nom du fichier depuis le chemin complet
# "/content/.../images/Abyssinian_1.jpg" → "Abyssinian_1.jpg"
df_meta = df_images.withColumn(
    "filename",
    F.element_at(F.split(F.col("path"), "/"), -1)
)

# Extraire le nom de la race (supprimer "_N.jpg" à la fin)
df_meta = df_meta.withColumn(
    "race_raw",
    F.regexp_replace(F.col("filename"), r"_\d+\.jpg$", "")
)

# Normaliser : remplacer les underscores par des espaces
df_meta = df_meta.withColumn(
    "race",
    F.regexp_replace(F.col("race_raw"), "_", " ")
)

# Extraire l'ID numérique
df_meta = df_meta.withColumn(
    "image_id",
    F.regexp_extract(F.col("filename"), r"_(\d+)\.jpg$", 1).cast("int")
)

# Détecter l'espèce : chats → majuscule (Abyssinian), chiens → minuscule (golden_retriever)
df_meta = df_meta.withColumn(
    "species",
    F.when(F.col("race_raw").rlike("^[A-Z][a-z]+"), "Cat")
     .otherwise("Dog")
)

# Calculer la taille en MB
df_meta = df_meta.withColumn(
    "size_mb",
    (F.col("length") / (1024 * 1024)).cast("decimal(10,2)")
)

# Toutes ces transformations sont LAZY — aucune image n'est lue pour l'instant
print("Pipeline de métadonnées créé (lazy) — aucune image traitée.")
df_meta.printSchema()

In [ ]:
# ACTION — Spark traite l'ensemble des images pour la première fois
total = df_meta.count()
print(f"✓ {total:,} images traitées")

# Afficher un échantillon des métadonnées extraites
df_meta.select("filename", "race", "species", "image_id", "size_mb").show(5, truncate=False)

In [ ]:
# Distribution chats vs chiens
df_meta.groupBy("species") \
    .agg(
        F.count("*").alias("nb_images"),
        F.avg("size_mb").alias("taille_moy_mb"),
        F.min("size_mb").alias("taille_min_mb"),
        F.max("size_mb").alias("taille_max_mb")
    ).show()

In [ ]:
# Top 10 races les plus représentées
df_meta.groupBy("race", "species") \
    .agg(F.count("*").alias("nb_images")) \
    .orderBy(F.col("nb_images").desc()) \
    .limit(10) \
    .show(truncate=False)

## 4. Décoder les images — UDF Pillow

Pour connaître les vraies dimensions `width × height`, il faut **décoder** chaque image depuis ses octets bruts (colonne `content`).

Une **UDF (User Defined Function)** permet d'appliquer n'importe quelle fonction Python sur chaque ligne du DataFrame, **en parallèle sur tous les workers**.

| Concept | Explication |
|---|---|
| **Sérialisation** | La fonction Python est convertie en bytes (`pickle`) et envoyée à chaque worker |
| **Parallélisme** | Chaque worker décode ses images indépendamment |
| **Retour struct** | Un `StructType` permet de retourner plusieurs valeurs (width, height, channels, format) |
| **Limite** | Les UDF Python sont 10-100× plus lentes que les fonctions natives Spark (sérialisation JVM ↔ Python) |

> On travaille sur un **échantillon de 300 images** pour rester rapide en Colab.  
> L'UDF est **lazy** — Spark ne décode rien avant une action.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from PIL import Image
import io

# Schéma de retour de l'UDF (structure avec 4 champs)
info_schema = StructType([
    StructField("width",    IntegerType(), True),
    StructField("height",   IntegerType(), True),
    StructField("channels", IntegerType(), True),
    StructField("format",   StringType(),  True)
])

@udf(returnType=info_schema)
def extract_image_info(image_bytes):
    """Extrait les dimensions d'une image depuis ses octets bruts (exécuté sur chaque worker)"""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        channels_map = {'RGB': 3, 'RGBA': 4, 'L': 1}
        channels = channels_map.get(img.mode, len(img.getbands()))
        return {
            'width':    img.width,
            'height':   img.height,
            'channels': channels,
            'format':   img.format
        }
    except Exception:
        return None  # Retourne None si l'image est corrompue

print("UDF définie — elle sera sérialisée et envoyée aux workers lors de la première action.")

In [ ]:
# Échantillon de 300 images pour rester rapide en Colab
df_sample = df_meta.limit(300)

# Application de l'UDF — LAZY (rien calculé)
df_dims_raw = df_sample.withColumn(
    "image_info",
    extract_image_info(F.col("content"))
)

# Aplatir le struct en colonnes séparées
df_dims = df_dims_raw \
    .withColumn("width",        F.col("image_info.width")) \
    .withColumn("height",       F.col("image_info.height")) \
    .withColumn("channels",     F.col("image_info.channels")) \
    .withColumn("format",       F.col("image_info.format")) \
    .withColumn("pixels",       F.col("width") * F.col("height")) \
    .withColumn("aspect_ratio", (F.col("width") / F.col("height")).cast("decimal(10,2)")) \
    .drop("image_info") \
    .cache()

# ACTION — Déclenche le décodage distribué des 300 images
# ⏳ Peut prendre 2-5 minutes selon la puissance de Colab
df_dims.count()  # matérialise le cache

df_dims.select("filename", "race", "width", "height", "channels", "format").show(10)
print(f"\nColonnes disponibles : {df_dims.columns}")

## 5. Dimensions — les images sont-elles toutes de la même taille ?

Dans un pipeline ML, les images doivent souvent être **normalisées** (même taille).  
Ici on explore la distribution des dimensions originales pour comprendre pourquoi un redimensionnement est nécessaire.

In [ ]:
# Statistiques sur les dimensions (min, max, moyenne, écart-type)
monitor.execute_and_monitor(
    lambda: df_dims.select("width", "height", "pixels")
                   .describe()
                   .show(),
    "stats dimensions"
)

In [ ]:
# Orientation des images (paysage / portrait / carré)
monitor.execute_and_monitor(
    lambda: df_dims.groupBy(
        F.when(F.col("width") > F.col("height"), "Paysage")
         .when(F.col("width") < F.col("height"), "Portrait")
         .otherwise("Carré").alias("orientation")
    ).count().orderBy("count", ascending=False).show(),
    "orientation images"
)

## 6. Partitionnement des données

Spark découpe automatiquement les fichiers en **partitions** lors du chargement.  
Chaque partition est traitée par un worker différent — plus il y a de partitions, plus le travail est distribué.

| Opération | Effet | Shuffle ? |
|---|---|---|
| `repartition(n)` | Redistribue les données en `n` partitions | ✅ Oui (coûteux) |
| `coalesce(n)` | Réduit le nombre de partitions | ❌ Non (économique) |

> **Règle pratique :** visez **2–4 partitions par CPU** disponible.  
> Sur Colab (2 cores), 4–8 partitions est un bon point de départ.

In [ ]:
# Voir le nombre de partitions actuelles
print(f"Partitions actuelles : {df_dims.rdd.getNumPartitions()}")

# repartition() → redistribue en N partitions (SHUFFLE — coûteux)
df_repartitioned = df_dims.repartition(8)
print(f"Après repartition(8) : {df_repartitioned.rdd.getNumPartitions()}")

# coalesce() → réduit le nombre de partitions SANS shuffle (économique)
df_coalesced = df_dims.coalesce(2)
print(f"Après coalesce(2)    : {df_coalesced.rdd.getNumPartitions()}")

## 7. Et le Machine Learning ?

Ce notebook a couvert l'ensemble du **Cours 4 — Traitement Distribué d'Images** :  
chargement `binaryFile`, extraction de métadonnées, stats, UDF Pillow, analyse des dimensions, redimensionnement et partitionnement.

**PySpark n'est pas un framework de Deep Learning**, mais il excelle pour la **préparation des données** :

| Ce que PySpark fait bien | Ce que PyTorch / TensorFlow fait bien |
|---|---|
| Téléchargement & extraction distribués | Entraînement du modèle (GPU) |
| Nettoyage, filtrage, redimensionnement | Transfer Learning (ResNet, EfficientNet…) |
| Export vers GCS / S3 / Azure Blob | Inférence sur de nouvelles images |

**Prochaines étapes possibles :**
- Exporter les images redimensionnées vers un stockage cloud (GCS, S3, Azure Blob)
- Utiliser `torch.utils.data.Dataset` pour charger les images préparées par Spark
- Entraîner un modèle de classification sur les 37 races (chats & chiens)